## 1. Naives bayes classifier

In [9]:
import os
import sys

sys.path.insert(0, os.path.abspath(os.path.join("..", "src")))

In [14]:
# --- BLOC 1 : charger le vrai dataset ---
from classical_ml.data_loader import load_movie_reviews
from classical_ml.naive_bayes_model import evaluate_model, train_naive_bayes

X_train, X_test, y_train, y_test = load_movie_reviews()
print("Taille train :", len(X_train), " | Taille test :", len(X_test))
print("Exemple d'avis (100 premiers caracteres) :", X_train[0][:200])

Taille train : 1600  | Taille test : 400
Exemple d'avis (100 premiers caracteres) : saving private ryan ( dreamworks ) running time : 2 hours 48 minutes . 
starring tom hanks , edward burns , tom sizemore and matt damon directed by steven spielberg already being hailed as the 'greate


In [ ]:
# --- BLOC 2 : entrainement et evaluation ---
model, vectorizer = train_naive_bayes(X_train, y_train)
resultats = evaluate_model(model, vectorizer, X_test, y_test)


print("\nAccuracy :", resultats["accuracy"])
print("\nRapport detaille :")
for classe, valeurs in resultats["report"].items():
    if isinstance(valeurs, dict):
        print(
            f"  {classe:12} precision={valeurs['precision']:.3f} j\
                recall={valeurs['recall']:.3f} f1={valeurs['f1-score']:.3f}"
        )


Accuracy : 0.8075

Rapport detaille :
  neg          precision=0.794 recall=0.830 f1=0.812
  pos          precision=0.822 recall=0.785 f1=0.803
  macro avg    precision=0.808 recall=0.807 f1=0.807
  weighted avg precision=0.808 recall=0.807 f1=0.807


In [21]:
# --- BLOC 3 : effet de max_features sur la performance ---
for taille in [500, 2000, 5000, 10000]:
    m, v = train_naive_bayes(X_train, y_train, max_features=taille)
    r = evaluate_model(m, v, X_test, y_test)
    print(f"max_features={taille:6} -> accuracy={r['accuracy']:.3f}")

max_features=   500 -> accuracy=0.757
max_features=  2000 -> accuracy=0.812
max_features=  5000 -> accuracy=0.807
max_features= 10000 -> accuracy=0.795


In [22]:
# --- BLOC 4 : inspecter les mots les plus "positifs" et "negatifs" selon le modele ---
import numpy as np

feature_names = np.array(vectorizer.get_feature_names_out())
log_probs = model.feature_log_prob_  # (nb_classes, nb_features)
classes = model.classes_
print("\nClasses dans l'ordre :", classes)

# difference de log-probabilite entre les deux classes, par mot
diff = log_probs[1] - log_probs[0]  # suppose classes[1] = 'pos', classes[0] = 'neg'
top_positifs = feature_names[np.argsort(diff)[-10:]]
top_negatifs = feature_names[np.argsort(diff)[:10]]

print("Mots les plus associes a 'pos' :", list(top_positifs))
print("Mots les plus associes a 'neg' :", list(top_negatifs))


Classes dans l'ordre : ['neg' 'pos']
Mots les plus associes a 'pos' : ['argento', 'hanks', 'lebowski', 'crowe', 'outstanding', 'damon', 'shrek', 'truman', 'flynt', 'mulan']
Mots les plus associes a 'neg' : ['seagal', 'worst', 'jawbreaker', 'waste', 'boring', 'ridiculous', 'schumacher', 'martha', 'stupid', 'lame']


In [25]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB

X_train = [
    "amazing great delivery",
    "amazing great delivery",
    "terrible worst delivery",
    "terrible worst delivery",
]
y_train = ["pos", "pos", "neg", "neg"]

vectorizer = TfidfVectorizer()
X_vec = vectorizer.fit_transform(X_train)
model = MultinomialNB()
model.fit(X_vec, y_train)

print("Vocabulaire (ordre des colonnes) :", list(vectorizer.get_feature_names_out()))
print("Classes apprises (ordre) :", model.classes_)
print()
print("feature_log_prob_ (shape) :", model.feature_log_prob_.shape)
print(model.feature_log_prob_)

Vocabulaire (ordre des colonnes) : ['amazing', 'delivery', 'great', 'terrible', 'worst']
Classes apprises (ordre) : ['neg' 'pos']

feature_log_prob_ (shape) : (2, 5)
[[-2.12936555 -1.51537334 -2.12936555 -1.30480943 -1.30480943]
 [-1.30480943 -1.51537334 -1.30480943 -2.12936555 -2.12936555]]


## 2. Logistic regression

In [3]:
# --- BLOC 1 : entrainement et comparaison directe avec Naive Bayes ---
import os
import sys

sys.path.insert(0, os.path.abspath(os.path.join("..", "src")))
from classical_ml.data_loader import load_movie_reviews
from classical_ml.logistic_regression import evaluate_model as eval_lr
from classical_ml.logistic_regression import train_logistic_regression
from classical_ml.naive_bayes_model import evaluate_model as eval_nb
from classical_ml.naive_bayes_model import train_naive_bayes

X_train, X_test, y_train, y_test = load_movie_reviews()

model_nb, vec_nb = train_naive_bayes(X_train, y_train)
resultats_nb = eval_nb(model_nb, vec_nb, X_test, y_test)

model_lr, vec_lr = train_logistic_regression(X_train, y_train)
resultats_lr = eval_lr(model_lr, vec_lr, X_test, y_test)

print(f"Naive Bayes         : {resultats_nb['accuracy']:.3f}")
print(f"Logistic Regression : {resultats_lr['accuracy']:.3f}")

Naive Bayes         : 0.807
Logistic Regression : 0.828


In [4]:
# --- BLOC 2 : effet du parametre C (regularisation) ---
from sklearn.metrics import accuracy_score

print("\n{:>8} {:>16} {:>16}".format("C", "accuracy TRAIN", "accuracy TEST"))
for C in [0.001, 0.01, 0.1, 1.0, 10, 100]:
    model, vectorizer = train_logistic_regression(X_train, y_train, C=C)
    X_train_vec = vectorizer.transform(X_train)
    acc_train = accuracy_score(y_train, model.predict(X_train_vec))
    resultats = eval_lr(model, vectorizer, X_test, y_test)
    print(f"{C:>8} {acc_train:>16.3f} {resultats['accuracy']:>16.3f}")


       C   accuracy TRAIN    accuracy TEST
   0.001            0.891            0.787
    0.01            0.887            0.792
     0.1            0.901            0.805
     1.0            0.962            0.828
      10            0.999            0.835
     100            1.000            0.830


In [ ]:
# --- BLOC 3 : inspecter les poids appris \
# (equivalent du feature_log_prob_ de Naive Bayes) ---
import numpy as np

model, vectorizer = train_logistic_regression(X_train, y_train, C=1.0)
feature_names = np.array(vectorizer.get_feature_names_out())
coefficients = model.coef_[0]  # un seul jeu de poids (classification binaire)

top_positifs = feature_names[np.argsort(coefficients)[-10:]]
top_negatifs = feature_names[np.argsort(coefficients)[:10]]

print(
    "\nMots avec le coefficient le plus POSITIF (poussent vers 'pos') :",
    list(top_positifs),
)
print(
    "Mots avec le coefficient le plus NEGATIF (poussent vers 'neg') :",
    list(top_negatifs),
)


Mots avec le coefficient le plus POSITIF (poussent vers 'pos') : ['mulan', 'overall', 'truman', 'perfect', 'best', 'family', 'excellent', 'war', 'life', 'great']
Mots avec le coefficient le plus NEGATIF (poussent vers 'neg') : ['bad', 'worst', 'plot', 'boring', 'movie', 'supposed', 'script', 'reason', 'waste', 'stupid']


## 3. SVM

In [9]:
# --- BLOC 1 : comparaison des 3 modeles vus jusqu'ici ---
import sys

sys.path.insert(0, os.path.abspath(os.path.join("..", "src")))
from classical_ml.data_loader import load_movie_reviews
from classical_ml.logistic_regression import evaluate_model as eval_lr
from classical_ml.logistic_regression import train_logistic_regression
from classical_ml.naive_bayes_model import evaluate_model as eval_nb
from classical_ml.naive_bayes_model import train_naive_bayes
from classical_ml.svm import evaluate_model as eval_svm
from classical_ml.svm import train_linear_svm

X_train, X_test, y_train, y_test = load_movie_reviews()

m_nb, v_nb = train_naive_bayes(X_train, y_train)
m_lr, v_lr = train_logistic_regression(X_train, y_train)
m_svm, v_svm = train_linear_svm(X_train, y_train)

print(f"Naive Bayes         : {eval_nb(m_nb, v_nb, X_test, y_test)['accuracy']:.3f}")
print(f"Logistic Regression : {eval_lr(m_lr, v_lr, X_test, y_test)['accuracy']:.3f}")
print(f"Linear SVM          : {eval_svm(m_svm, v_svm, X_test, y_test)['accuracy']:.3f}")

Naive Bayes         : 0.807
Logistic Regression : 0.828
Linear SVM          : 0.833


In [10]:
# --- BLOC 2 : effet du parametre C sur le SVM ---
from sklearn.metrics import accuracy_score

print("\n{:>8} {:>16} {:>16}".format("C", "accuracy TRAIN", "accuracy TEST"))
for C in [0.01, 0.1, 1.0, 10]:
    model, vectorizer = train_linear_svm(X_train, y_train, C=C)
    X_train_vec = vectorizer.transform(X_train)
    acc_train = accuracy_score(y_train, model.predict(X_train_vec))
    acc_test = eval_svm(model, vectorizer, X_test, y_test)["accuracy"]
    print(f"{C:>8} {acc_train:>16.3f} {acc_test:>16.3f}")


       C   accuracy TRAIN    accuracy TEST
    0.01            0.899            0.805
     0.1            0.957            0.828
     1.0            0.999            0.833
      10            1.000            0.833


In [11]:
# --- BLOC 3 : comparer les mots importants selon les 3 modeles ---
import numpy as np

feature_names_svm = np.array(v_svm.get_feature_names_out())
coefs_svm = m_svm.coef_[0]
top_pos_svm = feature_names_svm[np.argsort(coefs_svm)[-10:]]
top_neg_svm = feature_names_svm[np.argsort(coefs_svm)[:10]]

feature_names_lr = np.array(v_lr.get_feature_names_out())
coefs_lr = m_lr.coef_[0]
top_pos_lr = feature_names_lr[np.argsort(coefs_lr)[-10:]]
top_neg_lr = feature_names_lr[np.argsort(coefs_lr)[:10]]

print("\nTop mots 'pos' selon SVM :", list(top_pos_svm))
print("Top mots 'pos' selon LR  :", list(top_pos_lr))
print()
print("Top mots 'neg' selon SVM :", list(top_neg_svm))
print("Top mots 'neg' selon LR  :", list(top_neg_lr))

# A retenir : SVM et Logistic Regression apprennent tous deux un poids
# par mot (model.coef_), mais avec des objectifs d'optimisation
# differents -- les listes se recoupent souvent sans etre identiques.


Top mots 'pos' selon SVM : ['job', 'memorable', 'quite', 'terrific', 'perfect', 'excellent', 'hilarious', 'overall', 'fun', 'great']
Top mots 'pos' selon LR  : ['mulan', 'overall', 'truman', 'perfect', 'best', 'family', 'excellent', 'war', 'life', 'great']

Top mots 'neg' selon SVM : ['bad', 'plot', 'worst', 'supposed', 'unfortunately', 'boring', 'ridiculous', 'awful', 'looks', 'script']
Top mots 'neg' selon LR  : ['bad', 'worst', 'plot', 'boring', 'movie', 'supposed', 'script', 'reason', 'waste', 'stupid']


## 4. decision tree

## 5. random forest

In [ ]:
# --- BLOC 1 : entrainement de base ---
import sys

sys.path.insert(0, os.path.abspath(os.path.join("..", "src")))
from classical_ml.data_loader import load_movie_reviews
from classical_ml.random_forest import evaluate_model, train_random_forest

X_train, X_test, y_train, y_test = load_movie_reviews()

model, vectorizer = train_random_forest(X_train, y_train)
resultats = evaluate_model(model, vectorizer, X_test, y_test)
print("Accuracy :", resultats["accuracy"])
# Resultat attendu : ~0.792

In [ ]:
# --- BLOC 2 : la preuve du bagging -- foret vs arbre unique ---
from classical_ml.decision_tree_model import evaluate_model as eval_tree
from classical_ml.decision_tree_model import train_decision_tree

m_tree, v_tree = train_decision_tree(X_train, y_train)
acc_tree = eval_tree(m_tree, v_tree, X_test, y_test)["accuracy"]

print(f"\nArbre unique  : {acc_tree:.3f}")
print(f"Random Forest : {resultats['accuracy']:.3f}")
print(f"Gain du bagging : +{(resultats['accuracy'] - acc_tree) * 100:.1f} points")

In [ ]:
# --- BLOC 3 : effet du nombre d'arbres (n_estimators) ---
print(f"\n{'n_estimators':>14} {'accuracy TEST':>16}")
for n in [1, 5, 10, 50, 100, 200]:
    m, v = train_random_forest(X_train, y_train, n_estimators=n)
    acc = evaluate_model(m, v, X_test, y_test)["accuracy"]
    print(f"{n:>14} {acc:>16.3f}")

# A retenir : contrairement au boosting, ajouter des arbres au bagging
# ne provoque PAS de surapprentissage -- la performance monte puis
# plafonne. C'est une propriete rassurante du Random Forest.

In [ ]:
# --- BLOC 4 : mots les plus importants selon la foret ---
import numpy as np

noms = np.array(vectorizer.get_feature_names_out())
importances = model.feature_importances_
top = noms[np.argsort(importances)[-15:]]
print("\n15 mots les plus importants selon la foret :", list(top))

# Note : feature_importances_ mesure "combien ce mot a servi a separer
# les classes en moyenne sur tous les arbres", pas la DIRECTION du
# sentiment (contrairement aux coefficients signes de SVM/LogisticRegression).

In [ ]:
# --- BLOC 5 : train vs test -- la foret surapprend-elle ? ---
from sklearn.metrics import accuracy_score

acc_train = accuracy_score(y_train, model.predict(vectorizer.transform(X_train)))
print(f"\nAccuracy TRAIN : {acc_train:.3f}")
print(f"Accuracy TEST  : {resultats['accuracy']:.3f}")
print(f"Ecart : {(acc_train - resultats['accuracy']) * 100:.1f} points")

## 6. gradient_boosting

In [ ]:
# --- BLOC 1 : entrainement de base ---
import sys
import time

sys.path.insert(0, os.path.abspath(os.path.join("..", "src")))
from classical_ml.data_loader import load_movie_reviews
from classical_ml.gradient_boosting import evaluate_model, train_gradient_boosting

X_train, X_test, y_train, y_test = load_movie_reviews()

debut = time.time()
model, vectorizer = train_gradient_boosting(X_train, y_train)
duree = time.time() - debut
resultats = evaluate_model(model, vectorizer, X_test, y_test)
print(f"Accuracy : {resultats['accuracy']:.3f}  (entraine en {duree:.1f}s)")
# Resultat attendu : ~0.792 en ~6s

In [ ]:
# --- BLOC 2 : boosting vs bagging -- meme score, temps tres different ---
from classical_ml.random_forest import evaluate_model as eval_rf
from classical_ml.random_forest import train_random_forest

debut = time.time()
m_rf, v_rf = train_random_forest(X_train, y_train)
duree_rf = time.time() - debut
acc_rf = eval_rf(m_rf, v_rf, X_test, y_test)["accuracy"]

print(f"\nRandom Forest (bagging, parallelisable)  : {acc_rf:.3f} en {duree_rf:.1f}s")
print(f"Gradient Boosting (sequentiel) : {resultats['accuracy']:.3f} en {duree:.1f}s")
# A retenir : score comparable, mais le boosting est bien plus lent car
# chaque arbre depend du precedent -- impossible de paralleliser.

In [ ]:
# --- BLOC 3 : le boosting PEUT surapprendre (contrairement au bagging) ---
from sklearn.metrics import accuracy_score

print(f"\n{'n_estimators':>14} {'acc TRAIN':>12} {'acc TEST':>12}")
for n in [10, 50, 100, 300]:
    m, v = train_gradient_boosting(X_train, y_train, n_estimators=n)
    acc_train = accuracy_score(y_train, m.predict(v.transform(X_train)))
    acc_test = evaluate_model(m, v, X_test, y_test)["accuracy"]
    print(f"{n:>14} {acc_train:>12.3f} {acc_test:>12.3f}")

# A retenir : l'accuracy TRAIN monte continuellement avec le nombre
# d'arbres (chaque arbre corrige les erreurs restantes), mais l'accuracy
# TEST finit par stagner voire baisser -- c'est le surapprentissage
# propre au boosting, absent du bagging.

In [ ]:
# --- BLOC 4 : effet du learning_rate ---
print(f"\n{'learning_rate':>15} {'acc TRAIN':>12} {'acc TEST':>12}")
for lr in [0.01, 0.05, 0.1, 0.5]:
    m, v = train_gradient_boosting(X_train, y_train, n_estimators=50, learning_rate=lr)
    acc_train = accuracy_score(y_train, m.predict(v.transform(X_train)))
    acc_test = evaluate_model(m, v, X_test, y_test)["accuracy"]
    print(f"{lr:>15} {acc_train:>12.3f} {acc_test:>12.3f}")

# A retenir : learning_rate petit = corrections prudentes (besoin de plus
# d'arbres) ; grand = corrections agressives (converge vite, surapprend vite).
# Il y a un compromis entre learning_rate et n_estimators.

## 7.Xgboost

In [ ]:
# --- BLOC 1 : entrainement de base ---
import sys
import time

sys.path.insert(0, os.path.abspath(os.path.join("..", "src")))
from classical_ml.data_loader import load_movie_reviews
from classical_ml.xgboost import evaluate_model, train_xgboost

X_train, X_test, y_train, y_test = load_movie_reviews()

debut = time.time()
model, vectorizer, le = train_xgboost(X_train, y_train)
duree = time.time() - debut
resultats = evaluate_model(model, vectorizer, le, X_test, y_test)
print(f"Accuracy : {resultats['accuracy']:.3f}  (entraine en {duree:.1f}s)")
# Resultat attendu : ~0.825 en ~9s -- le meilleur des modeles a arbres

In [ ]:
# --- BLOC 2 : XGBoost vs GradientBoosting classique ---
from classical_ml.gradient_boosting import evaluate_model as eval_gb
from classical_ml.gradient_boosting import train_gradient_boosting

debut = time.time()
m_gb, v_gb = train_gradient_boosting(X_train, y_train)
duree_gb = time.time() - debut
acc_gb = eval_gb(m_gb, v_gb, X_test, y_test)["accuracy"]

print(f"\nGradientBoosting (sklearn) : {acc_gb:.3f} en {duree_gb:.1f}s")
print(f"XGBoost (optimise)          : {resultats['accuracy']:.3f} en {duree:.1f}s")
# A retenir : meme algorithme de fond, mais XGBoost ajoute regularisation
# L1/L2 + parallelisation de la construction de chaque arbre + gestion
# native des donnees creuses (notre cas avec TF-IDF).

In [ ]:
# --- BLOC 3 : effet de max_depth (profondeur des arbres) ---
from sklearn.metrics import accuracy_score

print(f"\n{'max_depth':>11} {'acc TRAIN':>12} {'acc TEST':>12}")
for d in [2, 4, 6, 10]:
    m, v, n = train_xgboost(X_train, y_train, max_depth=d)
    acc_train = accuracy_score(
        y_train, n.inverse_transform(m.predict(v.transform(X_train)))
    )
    acc_test = evaluate_model(m, v, n, X_test, y_test)["accuracy"]
    print(f"{d:>11} {acc_train:>12.3f} {acc_test:>12.3f}")

In [ ]:
# --- BLOC 4 : effet du learning_rate ---
print(f"\n{'learning_rate':>15} {'acc TEST':>12}")
for lr in [0.05, 0.1, 0.3, 0.5]:
    m, v, n = train_xgboost(X_train, y_train, learning_rate=lr)
    acc = evaluate_model(m, v, n, X_test, y_test)["accuracy"]
    print(f"{lr:>15} {acc:>12.3f}")

In [ ]:
# --- BLOC 5 : mots les plus importants ---
import numpy as np

noms = np.array(vectorizer.get_feature_names_out())
top = noms[np.argsort(model.feature_importances_)[-15:]]
print("\n15 mots les plus importants selon XGBoost :", list(top))

## 8.Lightgbm

In [ ]:
# --- BLOC 1 : entrainement de base ---
import sys
import time

sys.path.insert(0, "src")
from classical_ml.data_loader import load_movie_reviews
from classical_ml.lightgbm import evaluate_model, train_lightgbm

X_train, X_test, y_train, y_test = load_movie_reviews()

debut = time.time()
model, vectorizer, le = train_lightgbm(X_train, y_train)
duree = time.time() - debut
resultats = evaluate_model(model, vectorizer, le, X_test, y_test)
print(f"Accuracy : {resultats['accuracy']:.3f}  (entraine en {duree:.1f}s)")
# Resultat attendu : ~0.800 en ~3s

In [ ]:
# --- BLOC 2 : le compromis vitesse/precision face a XGBoost ---
from classical_ml.xgboost import evaluate_model as eval_xgb
from classical_ml.xgboost import train_xgboost

debut = time.time()
m_xgb, v_xgb, l_xgb = train_xgboost(X_train, y_train)
duree_xgb = time.time() - debut
acc_xgb = eval_xgb(m_xgb, v_xgb, l_xgb, X_test, y_test)["accuracy"]

print(f"\nLightGBM (leaf-wise) : {resultats['accuracy']:.3f} en {duree:.1f}s")
print(f"XGBoost (level-wise) : {acc_xgb:.3f} en {duree_xgb:.1f}s")
print(
    f"-> LightGBM est ~{duree_xgb / duree:.1f}x plus rapide, mais \
        {(acc_xgb - resultats['accuracy']) * 100:.1f} points moins precis"
)

In [ ]:
# --- BLOC 3 : effet de num_leaves (LE parametre cle de LightGBM) ---
from sklearn.metrics import accuracy_score

print(f"\n{'num_leaves':>12} {'acc TRAIN':>12} {'acc TEST':>12}")
for n in [5, 15, 31, 63, 127]:
    m, v, n = train_lightgbm(X_train, y_train, num_leaves=n)
    acc_train = accuracy_score(
        y_train, n.inverse_transform(m.predict(v.transform(X_train)))
    )
    acc_test = evaluate_model(m, v, n, X_test, y_test)["accuracy"]
    print(f"{n:>12} {acc_train:>12.3f} {acc_test:>12.3f}")

# A retenir : num_leaves remplace max_depth chez LightGBM. Comme la
# croissance est leaf-wise (arbres desequilibres), c'est ce plafond qui
# controle la complexite -- augmenter num_leaves fait vite surapprendre.

In [ ]:
# --- BLOC 4 : effet du nombre d'arbres ---
print(f"\n{'n_estimators':>14} {'acc TEST':>12} {'temps (s)':>12}")
for n in [20, 50, 100, 300]:
    debut = time.time()
    m, v, n = train_lightgbm(X_train, y_train, n_estimators=n)
    d = time.time() - debut
    acc = evaluate_model(m, v, n, X_test, y_test)["accuracy"]
    print(f"{n:>14} {acc:>12.3f} {d:>12.1f}")

In [ ]:
# --- BLOC 5 : mots les plus importants ---
import numpy as np

noms = np.array(vectorizer.get_feature_names_out())
top = noms[np.argsort(model.feature_importances_)[-15:]]
print("\n15 mots les plus importants selon LightGBM :", list(top))

## 9.Catboost

In [ ]:
# ATTENTION : avec les parametres par defaut (max_features=2000,
# iterations=100), l'entrainement prend ~220 SECONDES sur ce dataset.
# Les blocs ci-dessous utilisent des parametres reduits par defaut --
# augmente-les seulement si tu es pret a attendre.

# --- BLOC 1 : entrainement rapide (parametres reduits) ---
import sys
import time

sys.path.insert(0, "src")
from classical_ml.catboost import evaluate_model, train_catboost
from classical_ml.data_loader import load_movie_reviews

X_train, X_test, y_train, y_test = load_movie_reviews()

debut = time.time()
model, vectorizer, le = train_catboost(
    X_train, y_train, max_features=500, iterations=50
)
duree = time.time() - debut
resultats = evaluate_model(model, vectorizer, le, X_test, y_test)
print(f"Accuracy (parametres reduits) : {resultats['accuracy']:.3f}  en {duree:.1f}s")

In [ ]:
# --- BLOC 2 : le vrai benchmark complet (LENT -- ~4 minutes) ---
# decommente pour lancer :
# debut = time.time()
# m_full, v_full, l_full = train_catboost(X_train, y_train)
# defauts : 2000 features, 100 iterations
# duree_full = time.time() - debut
# acc_full = evaluate_model(m_full, v_full, l_full, X_test, y_test)["accuracy"]
# print(f"Accuracy (parametres complets) : {acc_full:.3f} en {duree_full:.1f}s")
# Resultat observe pendant l'apprentissage : 0.823 en 220s


# --- BLOC 3 : comparaison avec les autres boosters ---
from classical_ml.lightgbm import evaluate_model as eval_lgb
from classical_ml.lightgbm import train_lightgbm
from classical_ml.xgboost import evaluate_model as eval_xgb
from classical_ml.xgboost import train_xgboost

debut = time.time()
m_x, v_x, l_x = train_xgboost(X_train, y_train)
d_x = time.time() - debut
acc_x = eval_xgb(m_x, v_x, l_x, X_test, y_test)["accuracy"]

debut = time.time()
m_l, v_l, l_l = train_lightgbm(X_train, y_train)
d_l = time.time() - debut
acc_l = eval_lgb(m_l, v_l, l_l, X_test, y_test)["accuracy"]

print(f"\n{'modele':12} {'accuracy':>10} {'temps (s)':>12}")
print(f"{'XGBoost':12} {acc_x:>10.3f} {d_x:>12.1f}")
print(f"{'LightGBM':12} {acc_l:>10.3f} {d_l:>12.1f}")
print(
    f"{'CatBoost':12} {'0.823':>10} {'220.0':>12}  (valeurs observees, params complets)"
)

# CONCLUSION : CatBoost est ~23x plus lent que XGBoost pour un score
# legerement INFERIEUR sur ce dataset. Son avantage principal (gestion
# native des variables CATEGORIELLES brutes) ne sert a rien ici, nos
# features TF-IDF etant deja numeriques et creuses.

In [ ]:
# --- BLOC 4 : effet de depth (arbres symetriques) ---
from sklearn.metrics import accuracy_score

print(f"\n{'depth':>7} {'acc TEST':>12} {'temps (s)':>12}")
for d in [2, 4, 6]:
    debut = time.time()
    m, v, n = train_catboost(X_train, y_train, max_features=500, iterations=50, depth=d)
    duree_d = time.time() - debut
    acc = evaluate_model(m, v, n, X_test, y_test)["accuracy"]
    print(f"{d:>7} {acc:>12.3f} {duree_d:>12.1f}")

# Rappel : CatBoost construit des arbres SYMETRIQUES -- a chaque niveau,
# tous les noeuds utilisent la meme condition de decoupe. Cela limite
# l'expressivite par arbre mais accelere l'inference et reduit le
# surapprentissage.